# Phase 3.2: Mechanism Ablations - Nonstationarity

Analyze how nonstationarity mechanisms affect c-GC and c-GC* recovery across depths.

## Scenarios
- **time_varying_coefficients**: Coefficient dynamics (VAR with time-varying A matrices)
- **regime_shift**: Discrete regime switches (multiple stationary regimes with transitions)

For each scenario:
- Run c-GC and c-GC* across depths [1,2,3,4,5,6]
- Compute recovery metrics (accuracy, precision, recall, FPR)
- Track D_p trajectories and instability signatures
- Include stationarity diagnostics
- Export results.csv, summary.json, figures/

## Setup: Imports and Configuration

In [ ]:
from __future__ import annotations

import sys
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Find project root
CAUSALISED_GC_RELATIVE_PATH = Path('src/markovianity_diagnostic/core/causalised-GC.py')
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / CAUSALISED_GC_RELATIVE_PATH).exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find causalised-GC.py')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from markovianity_diagnostic.experiments.simulations import (
    scenario_time_varying_coefficients,
    scenario_regime_shift,
)
from markovianity_diagnostic.experiments.adapters import METHODS
from markovianity_diagnostic.experiments.graph_metrics import summarize_run

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'simulations' / 'mechanism_ablations' / 'nonstationarity'
FIGURES_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
REPEATS = 10
T = 2000
D = 10
P_VALUES = [1, 2, 3, 4, 5, 6]
METHODS_TO_TEST = ['gcstar_cgc', 'gcstar_cgc_star']

SCENARIOS = [
    ('time_varying_coefficients', {'T': T, 'd': D, 'noise_scale': 1.0}),
    ('regime_shift', {'T': T, 'd': D, 'noise_scale': 1.0}),
]

print(f'Output directory: {OUTPUT_DIR}')

## Run Experiments

In [ ]:
scenario_functions = {
    'time_varying_coefficients': scenario_time_varying_coefficients,
    'regime_shift': scenario_regime_shift,
}

all_results = {}

for scenario_name, scenario_kwargs in SCENARIOS:
    print(f'Scenario: {scenario_name}')
    scenario_fn = scenario_functions[scenario_name]
    all_results[scenario_name] = {}
    
    for method_name in METHODS_TO_TEST:
        if method_name not in METHODS:
            continue
        analyze_fn = METHODS[method_name]
        method_results = []
        
        for repeat_idx in range(REPEATS):
            sample = scenario_fn(**scenario_kwargs, seed=SEED + repeat_idx)
            X = sample.X
            adjacencies_by_p = analyze_fn(X, P_VALUES)
            run_summary = summarize_run(
                adjacencies=adjacencies_by_p,
                ground_truth_compact=sample.ground_truth_compact,
                p_values=P_VALUES,
            )
            method_results.append({'repeat': repeat_idx, 'summary': run_summary})
        
        all_results[scenario_name][method_name] = method_results
        print(f'  ✓ {method_name}')

print('✓ All experiments completed')

## Export Results and Figures

In [ ]:
summary_rows = []
for scenario_name, method_results in all_results.items():
    for method_name, results in method_results.items():
        T_obs_values = [r['summary'].get('T_obs', 0.0) for r in results]
        row = {
            'scenario': scenario_name,
            'method': method_name,
            'repeats': len(results),
            'mean_T_obs': float(np.mean(T_obs_values)) if T_obs_values else 0.0,
        }
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_csv_path = OUTPUT_DIR / 'results.csv'
summary_df.to_csv(summary_csv_path, index=False)
print(f'Exported: {summary_csv_path}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for scenario_idx, (scenario_name, method_results) in enumerate(all_results.items()):
    ax = axes[scenario_idx]
    for method_name, results in method_results.items():
        d_means = {}
        for p_val in P_VALUES[1:]:
            d_values = [r['summary'].get('D_p', {}).get(str(p_val)) for r in results]
            d_values = [v for v in d_values if v is not None]
            d_means[p_val] = np.mean(d_values) if d_values else 0.0
        p_vals_sorted = sorted(d_means.keys())
        d_vals_sorted = [d_means[p] for p in p_vals_sorted]
        style = '-o' if method_name == 'gcstar_cgc' else '--s'
        ax.plot(p_vals_sorted, d_vals_sorted, style, label=method_name, linewidth=2)
    ax.set_xlabel('Conditioning Depth p')
    ax.set_ylabel('Mean D_p')
    ax.set_title(scenario_name)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'dp_trajectories.png', dpi=100, bbox_inches='tight')
print('Exported: dp_trajectories.png')
plt.close()

## Manifest

In [ ]:
manifest = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'analysis': 'mechanism_ablations_nonstationarity',
    'scenarios': list(all_results.keys()),
    'methods': METHODS_TO_TEST,
}

with open(OUTPUT_DIR / 'manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)

print('✓ All outputs verified successfully')